In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [2]:
!pip install roboflow

from roboflow import Roboflow
rf = Roboflow(api_key="TE2gtVnw3YrYI4zH8Mcb")
project = rf.workspace("v-0mqey").project("traffic-imvd8")
version = project.version(5)
dataset = version.download("yolov8")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 169.5/169.5 kB 5.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 66.8/66.8 kB 5.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.9/49.9 MB 39.3 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 69.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.5/5.5 MB 110.1 MB/s eta 0:00:0000:01
  Attempting uninstall: opencv-python-headless
    Found existing installation: opencv-python-headless 4.13.0.92
    Uninstalling opencv-python-headless-4.13.0.92:
      Successfully uninstalled opencv-python-headless-4.13.0.92
  Attempting uninstall: idna
    Found existing installation: idna 3.11
    Uninstalling idna-3.11:
      Successfully uninstalled idna-3.11
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-adk 1.25.1 requires google-cloud-bigquery


Extracting Dataset Version Zip to traffic-5 in yolov8:: 100%|██████████| 10568/10568 [00:01<00:00, 8307.27it/s]


In [3]:
import os
import shutil
import random
from collections import defaultdict

# ===============================
# Config
# ===============================
IMG_DIR = "/kaggle/working/traffic-5/train/images"      # original images
LBL_DIR = "/kaggle/working/traffic-5/train/labels"      # original YOLO labels
OUT_DIR = "/kaggle/working/dataset_splits"      # new split dataset

splits = {"train": 0.8, "valid": 0.1, "test": 0.1}

# Make output dirs
for split in splits:
    os.makedirs(os.path.join(OUT_DIR, split, "images"), exist_ok=True)
    os.makedirs(os.path.join(OUT_DIR, split, "labels"), exist_ok=True)

# ===============================
# Group images by class
# ===============================
class_to_images = defaultdict(list)

for lbl_file in os.listdir(LBL_DIR):
    if not lbl_file.endswith(".txt"):
        continue

    lbl_path = os.path.join(LBL_DIR, lbl_file)
    with open(lbl_path, "r") as f:
        lines = f.readlines()

    if not lines:
        continue

    # Take first class ID (assuming single-class per image)
    cls_id = int(lines[0].split()[0])

    if cls_id == 0:
        cname = "green"
    elif cls_id == 1:
        cname = "red"
    elif cls_id == 2:
        cname = "yellow"
    else:
        continue

    img_file = lbl_file.replace(".txt", ".jpg")  # change if your images are .png
    img_path = os.path.join(IMG_DIR, img_file)

    if os.path.exists(img_path):
        class_to_images[cname].append((img_path, lbl_path))

# ===============================
# Stratified Split
# ===============================
for cname, items in class_to_images.items():
    random.shuffle(items)
    n = len(items)

    n_train = int(n * splits["train"])
    n_valid = int(n * splits["valid"])
    n_test  = n - n_train - n_valid

    split_data = {
        "train": items[:n_train],
        "valid": items[n_train:n_train+n_valid],
        "test":  items[n_train+n_valid:]
    }

    # Copy files
    for split, pairs in split_data.items():
        for img_path, lbl_path in pairs:
            shutil.copy(img_path, os.path.join(OUT_DIR, split, "images", os.path.basename(img_path)))
            shutil.copy(lbl_path, os.path.join(OUT_DIR, split, "labels", os.path.basename(lbl_path)))

    print(f"{cname}: Train={len(split_data['train'])}, Valid={len(split_data['valid'])}, Test={len(split_data['test'])}")

print("\n✅ Stratified 80/10/10 split completed at:", OUT_DIR)

# ===============================
# Write new data.yaml
# ===============================
yaml_content = """names:
- green
- red
- yellow
nc: 3
train: {}/train/images
val: {}/valid/images
test: {}/test/images
""".format(OUT_DIR, OUT_DIR, OUT_DIR)

with open(os.path.join(OUT_DIR, "data.yaml"), "w") as f:
    f.write(yaml_content)

print("✅ data.yaml saved at:", os.path.join(OUT_DIR, "data.yaml"))

green: Train=1484, Valid=185, Test=186
red: Train=1431, Valid=178, Test=180
yellow: Train=1309, Valid=163, Test=165

✅ Stratified 80/10/10 split completed at: /kaggle/working/dataset_splits
✅ data.yaml saved at: /kaggle/working/dataset_splits/data.yaml


In [4]:
# =============================
# YOLOv8 Memory-Safe Training
# =============================
!pip install --upgrade ultralytics -q

from ultralytics import YOLO

# -------------------------
# Config
# -------------------------
DATA_YAML    = "/kaggle/working/dataset_splits/data.yaml"
BASE_WEIGHTS = "yolov8s.pt"  # Standard small model
BATCH        = 4             # Smaller per-GPU batch size
ACCUM        = 2             # Gradient accumulation → effective batch 8
WORKERS      = 2             # Fewer data loader workers
CACHE        = False         # Disable dataset caching in RAM
IMG_SIZE     = 640           # Full resolution
EPOCHS       = 30            # Number of epochs

# -------------------------
# Load model with transfer learning
# -------------------------
model = YOLO(BASE_WEIGHTS)

# -------------------------
# Train
# -------------------------
model.train(
    data=DATA_YAML,
    imgsz=IMG_SIZE,
    epochs=EPOCHS,
    batch=BATCH,
    device=0,
    amp=True,           # Mixed precision
    workers=WORKERS,
    cache=CACHE,        # Avoid RAM exhaustion
    optimizer="AdamW",  # Stable optimizer
    lr0=0.001,          # Base learning rate
    lrf=0.01,           # Final LR factor
    cos_lr=True,        # Cosine decay schedule
    warmup_epochs=1,    # Warmup
    patience=10         # Early stopping
)

# -------------------------
# Validate
# -------------------------
metrics = model.val()
print(metrics)

# -------------------------
# Inference Speed Test
# -------------------------
import time, glob
imgs = glob.glob("path/to/test/images/*.jpg")[:50]
start = time.time()
for img in imgs:
    _ = model(img)
fps = len(imgs) / (time.time() - start)
print(f"FPS: {fps:.2f}")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 17.9 MB/s eta 0:00:00 0:00:01
Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
Ultralytics 8.4.34 🚀 Python-3.12.12 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=4, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=True, cutmix=0.0, data=/kaggle/working/dataset_splits/data.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=30, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscri

In [5]:
metrics = model.val()  # Validation returns a Results object

# The Results object has properties like:
# metrics.box: contains metrics for bounding boxes
# metrics.box.precision - not directly accessible; instead, use metrics.box.map or metrics.key to find available.

# Convert Results to dictionary to inspect keys and values for metrics
# metrics_dict = metrics.metrics or metrics.box or metrics.value or {} # This line caused the error

print("\n=== YOLOv8 Evaluation Metrics ===")

# Example printout:
# You may need to adjust keys based on what your version returns

print(f"mAP@0.5:0.95  : {metrics.box.map:.4f}" if hasattr(metrics.box, "map") else "mAP@0.5:0.95  : N/A")
print(f"mAP@0.5       : {metrics.box.map50:.4f}" if hasattr(metrics.box, "map50") else "mAP@0.5       : N/A")

# Precision, Recall, F1 may be available as ~metrics.box.pr, metrics.box.recall, metrics.box.f1 or within dictionary keys

# If not available, access as:
# The error message indicates that 'pr', 're', and 'f1' might not be direct attributes.
# Let's try accessing them through the results_dict or mean_results if available.
# Based on the traceback, mean_results is a method that returns precision, recall, mAP50, and mAP50-95.
# We can also access the metrics directly from the 'box' attribute as shown in the traceback.

# Accessing precision, recall, and F1 from the 'box' attribute if available
if hasattr(metrics.box, "p") and hasattr(metrics.box, "r"):
    # Access the mean values from the arrays
    print(f"Precision    : {metrics.box.p.mean():.4f}")
    print(f"Recall       : {metrics.box.r.mean():.4f}")
    # Calculate F1 score from the mean precision and recall
    p_mean = metrics.box.p.mean()
    r_mean = metrics.box.r.mean()
    f1 = 2 * (p_mean * r_mean) / (p_mean + r_mean) if (p_mean + r_mean) > 0 else 0
    print(f"F1 Score     : {f1:.4f}")
else:
    # If not available as direct attributes, we can try accessing the mean_results
    try:
        p, r, map50, map = metrics.mean_results()
        print(f"Precision    : {p:.4f}")
        print(f"Recall       : {r:.4f}")
        # F1 score needs to be calculated manually if not directly available
        f1 = 2 * (p * r) / (p + r) if (p + r) > 0 else 0
        print(f"F1 Score     : {f1:.4f}")
    except Exception as e:
        print("Precision    : N/A")
        print("Recall       : N/A")
        print("F1 Score     : N/A")
        print(f"Could not calculate P, R, F1: {e}")

Ultralytics 8.4.34 🚀 Python-3.12.12 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 1106.6±376.8 MB/s, size: 54.7 KB)
val: Scanning /kaggle/working/dataset_splits/valid/labels.cache... 526 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 526/526 169.7Mit/s 0.0s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 33/33 3.7it/s 8.9s0.2s
                   all        526       1292       0.92       0.91       0.94      0.596
                 green        203        394      0.926      0.924      0.956       0.62
                   red        225        503      0.946      0.944      0.971      0.685
                yellow        192        395      0.888      0.863      0.895      0.484
Speed: 1.4ms preprocess, 9.8ms inference, 0.0ms loss, 2.0ms postprocess per image
Results saved to /kaggle/working/runs/detect/val2

=== YOLOv8 Evaluation Metrics ===
mAP@0.5:0.95  : 0.5962
mAP@0.5  

In [6]:
!mv /kaggle/working/runs/detect/train/weights/best.pt /kaggle/working/

In [7]:
!rm -rf /kaggle/working/runs/train

In [10]:
from ultralytics import YOLO

# Load the previously trained best model
model = YOLO("/kaggle/working/best.pt")

# Continue training for additional epochs
model.train(
    data="/kaggle/working/dataset_splits/data.yaml",
    imgsz=640,
    epochs=100,      # new additional epochs
    batch=4,
    device=0,
    amp=True,
    workers=2,
    cache=False,
    optimizer="AdamW",
    lr0=0.0005,      # optionally reduce learning rate
    lrf=0.01,
    cos_lr=True,
    warmup_epochs=1,
    patience=10
)

Ultralytics 8.4.34 🚀 Python-3.12.12 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=4, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=True, cutmix=0.0, data=/kaggle/working/dataset_splits/data.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=100, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.0005, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=/kaggle/working/best.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=train4, nbs=64, nms=False, opset=None, optimize=False, optimizer=AdamW, overlap_mask=True, p

ultralytics.utils.metrics.DetMetrics object with attributes:

ap_class_index: array([0, 1, 2])
box: ultralytics.utils.metrics.Metric object
confusion_matrix: <ultralytics.utils.metrics.ConfusionMatrix object at 0x7d5abc07b9b0>
curves: ['Precision-Recall(B)', 'F1-Confidence(B)', 'Precision-Confidence(B)', 'Recall-Confidence(B)']
curves_results: [[array([          0,    0.001001,    0.002002,    0.003003,    0.004004,    0.005005,    0.006006,    0.007007,    0.008008,    0.009009,     0.01001,    0.011011,    0.012012,    0.013013,    0.014014,    0.015015,    0.016016,    0.017017,    0.018018,    0.019019,     0.02002,    0.021021,    0.022022,    0.023023,
          0.024024,    0.025025,    0.026026,    0.027027,    0.028028,    0.029029,     0.03003,    0.031031,    0.032032,    0.033033,    0.034034,    0.035035,    0.036036,    0.037037,    0.038038,    0.039039,     0.04004,    0.041041,    0.042042,    0.043043,    0.044044,    0.045045,    0.046046,    0.047047,
          0.04

In [11]:
metrics = model.val()  # Validation returns a Results object

# The Results object has properties like:
# metrics.box: contains metrics for bounding boxes
# metrics.box.precision - not directly accessible; instead, use metrics.box.map or metrics.key to find available.

# Convert Results to dictionary to inspect keys and values for metrics
# metrics_dict = metrics.metrics or metrics.box or metrics.value or {} # This line caused the error

print("\n=== YOLOv8 Evaluation Metrics ===")

# Example printout:
# You may need to adjust keys based on what your version returns

print(f"mAP@0.5:0.95  : {metrics.box.map:.4f}" if hasattr(metrics.box, "map") else "mAP@0.5:0.95  : N/A")
print(f"mAP@0.5       : {metrics.box.map50:.4f}" if hasattr(metrics.box, "map50") else "mAP@0.5       : N/A")

# Precision, Recall, F1 may be available as ~metrics.box.pr, metrics.box.recall, metrics.box.f1 or within dictionary keys

# If not available, access as:
# The error message indicates that 'pr', 're', and 'f1' might not be direct attributes.
# Let's try accessing them through the results_dict or mean_results if available.
# Based on the traceback, mean_results is a method that returns precision, recall, mAP50, and mAP50-95.
# We can also access the metrics directly from the 'box' attribute as shown in the traceback.

# Accessing precision, recall, and F1 from the 'box' attribute if available
if hasattr(metrics.box, "p") and hasattr(metrics.box, "r"):
    # Access the mean values from the arrays
    print(f"Precision    : {metrics.box.p.mean():.4f}")
    print(f"Recall       : {metrics.box.r.mean():.4f}")
    # Calculate F1 score from the mean precision and recall
    p_mean = metrics.box.p.mean()
    r_mean = metrics.box.r.mean()
    f1 = 2 * (p_mean * r_mean) / (p_mean + r_mean) if (p_mean + r_mean) > 0 else 0
    print(f"F1 Score     : {f1:.4f}")
else:
    # If not available as direct attributes, we can try accessing the mean_results
    try:
        p, r, map50, map = metrics.mean_results()
        print(f"Precision    : {p:.4f}")
        print(f"Recall       : {r:.4f}")
        # F1 score needs to be calculated manually if not directly available
        f1 = 2 * (p * r) / (p + r) if (p + r) > 0 else 0
        print(f"F1 Score     : {f1:.4f}")
    except Exception as e:
        print("Precision    : N/A")
        print("Recall       : N/A")
        print("F1 Score     : N/A")
        print(f"Could not calculate P, R, F1: {e}")

Ultralytics 8.4.34 🚀 Python-3.12.12 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
Model summary (fused): 73 layers, 11,126,745 parameters, 0 gradients, 28.4 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 1202.4±698.3 MB/s, size: 55.5 KB)
val: Scanning /kaggle/working/dataset_splits/valid/labels.cache... 526 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 526/526 200.6Mit/s 0.0s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 33/33 3.7it/s 8.9s0.2s
                   all        526       1292      0.926      0.916      0.953      0.615
                 green        203        394      0.929      0.927      0.962      0.639
                   red        225        503      0.961      0.944      0.982      0.698
                yellow        192        395      0.888      0.878      0.916      0.507
Speed: 1.4ms preprocess, 9.7ms inference, 0.0ms loss, 1.9ms postprocess per image
Results saved to /kaggle/working/ru